### Supervisor

Multi-Agent 시스템의 구성 방식 중 하나인 Supervisor 패턴을 살펴본다.

In [1]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

MODEL_NAME = "gemini-3.6-flash"
llm = ChatGoogleGenerativeAI(model=MODEL_NAME)
search_llm = llm.bind_tools([{"google_search": {}}])

### Multi-Agent란

Multi-Agent 시스템은 하나의 Agent가 모든 작업을 처리하는 대신, 서로 다른 역할을 맡은 여러 Agent가 협력하여 하나의 목표를 달성하는 구조이다. 각 Agent는 역할에 맞는 Prompt, Tool, Context를 독립적으로 가질 수 있으며, Agent 사이의 작업 전달과 결과 공유를 통해 복잡한 작업을 나누어 처리한다.

### Multi-Agent를 고려하는 이유

하나의 Agent가 너무 많은 역할, Tool, Context를 담당하면 지시 충돌과 Tool 선택 오류가 늘어날 수 있다. 역할별 Agent로 분리하면 각 Agent가 자신의 작업에 필요한 지시와 정보에 집중할 수 있다.

| 이점 | 설명 |
|------|------|
| 역할과 Context 분리 | 각 Agent가 담당 작업에 필요한 정보에 집중 |
| Tool과 권한 분리 | Agent별로 필요한 Tool만 제공 |
| 독립적인 설계와 평가 | 역할마다 Prompt, 모델, 평가 기준을 다르게 설정 |
| 동적인 작업 위임 | 요청과 중간 결과에 따라 적절한 Agent를 선택 |

역할이 단순하고 실행 순서가 고정되어 있다면 단일 Agent나 일반 Workflow가 더 적합하다. Multi-Agent는 모델 호출 수, 지연 시간, 비용과 실패 지점을 증가시키므로 역할 분리와 동적 위임의 이점이 충분할 때 사용한다.

### Supervisor 패턴

Supervisor 패턴은 중앙의 Supervisor가 작업을 분석하여 적절한 Subagent에 분배하고, 각 결과를 종합해 최종 응답을 만드는 구조이다.

```text
사용자 → Supervisor Agent
             ├─→ Research Agent
             ├─→ Writer Agent
             └─→ Reviewer Agent
```

### Supervisor 구현

보고서 작성 과정을 자료 조사, 작성, 검토로 나눈다. `create_agent`로 만든 각 Subagent를 `@tool` 함수로 감싸 Supervisor의 Tool로 등록한다. 각 Tool은 작업 설명을 Subagent에 전달하고 마지막 응답만 Supervisor에 반환한다. Checkpointer는 전체 대화를 관리하는 최상위 Supervisor에만 설정한다.

In [2]:
researcher_agent = create_agent(
    model=search_llm,
    tools=[],
    system_prompt=(
        "너는 자료 조사 전문가다. 요청받은 주제를 Google Search로 조사해. "
        "보고서 작성에 필요한 핵심 사실과 근거를 정리하고, 각 내용의 출처를 함께 제시해."
    ),
    name="researcher",
)

writer_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "너는 보고서 작성 전문가다. 전달받은 조사 결과만을 근거로 보고서를 작성해. "
        "보고서는 제목, 개요, 본문, 결론 순서로 구성하고 출처를 유지해."
    ),
    name="writer",
)

reviewer_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "너는 보고서 검토 전문가다. 전달받은 보고서의 구조, 가독성, 내용의 일관성, "
        "출처 표기를 검토해. 수정이 필요하면 구체적인 수정 사항을 제시하고, "
        "문제가 없으면 '승인'이라고 명시해."
    ),
    name="reviewer",
)

In [3]:
@tool
def research(query: str) -> str:
    """주제에 관한 최신 자료와 출처를 조사한다."""
    result = researcher_agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].text


@tool
def write_report(query: str) -> str:
    """조사 결과를 바탕으로 보고서를 작성한다."""
    result = writer_agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].text


@tool
def review_report(query: str) -> str:
    """보고서를 검토하고 수정 사항이나 승인 여부를 반환한다."""
    result = reviewer_agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].text

In [4]:
supervisor_agent = create_agent(
    model=llm,
    tools=[research, write_report, review_report],
    system_prompt=(
        "너는 보고서 작성 팀의 Supervisor다. "
        "사용자의 요청을 분석하고 필요한 작업을 적절한 전문가에게 위임해. "
        "작업을 위임할 때는 해당 전문가에게 필요한 맥락을 충분히 전달해. "
        "전문가의 결과를 검토하고 필요한 경우 추가 작업을 요청해. "
        "근거와 출처가 포함된 완성도 높은 보고서를 최종 응답으로 제공해. "
        "중간 작업 과정은 사용자에게 설명하지 마."
    ),
    checkpointer=InMemorySaver(),
)

### 실행

In [5]:
config = {"configurable": {"thread_id": "report-session-1"}}

for event in supervisor_agent.stream(
    {"messages": [HumanMessage(content="AI Agent 기술 동향에 대한 보고서를 작성해줘.")]},
    config=config,
    stream_mode="updates",
):
    for node_name, state in event.items():
        message = state["messages"][-1]
        print("=" * 60)
        print(f"Node: {node_name}")
        if getattr(message, "tool_calls", None):
            print(f"Tool calls: {[call['name'] for call in message.tool_calls]}")
        if message.text:
            print(message.text[:500])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Node: model
Tool calls: ['research']
Node: tools
AI 에이전트(AI Agent) 및 자율형·LLM 기반 에이전트 분야의 최신 기술 동향, 주요 기업 사례, 시장 전망 보고서입니다.

---

### 1. 개요: 생성형 AI에서 '에이전틱 AI(Agentic AI)'로의 전환
* **개념 변화**: 기존 생성형 AI가 "질문-답변" 중심의 수동적 도구였다면, 자율형 AI 에이전트는 **"목표 부여-계획-실행-피드백"**을 자율 순환하여 작업을 완수하는 **'실행형 지능(Action-oriented AI)'**으로 진화했습니다.
* **챗봇과의 차이점**:
  * **기존 LLM/챗봇**: 단발성 텍스트 생성에 의존하며, 도구 활용 및 후속 작업은 인간이 직접 수행.
  * **AI 에이전트**: 목표(Goal)가 설정되면 스스로 워크플로우를 분할하고, 외부 API/도구 연동, 데이터 조회, 다단계 의사결정 및 최종 실행까지 인간의 개입을 최소화하여 완료.

---

### 2. LLM 기반 에이전트의 핵심 기술 아키텍처 및 특징
L
Node: model
Tool calls: ['write_report']
Node: tools
# [보고서] AI Agent 기술 동향 보고서: 생성형 AI에서 에이전틱 AI로의 진화와 시장 전망

---

## 1. 개요 (Introduction)

최근 인공지능 기술 생태계는 단순 대화 및 콘텐츠 생성을 넘어 스스로 목표를 설정하고 수행하는 **'에이전틱 AI(Agentic AI)'**로 빠르게 전환되고 있다. 기존의 대화형 거대언어모델(LLM)이 사용자의 요청에 응답하는 수동적 방식에 머물렀다면, 에이전틱 AI는 **실행 지향적 AI(Action-oriented AI)**로서 차별성을 가진다.

에이전트 기술은 **[목표 부여 → 계획 수립 → 실행 → 피드백]**의 자율적인 순환 과정을 통해 고도화된 업무를 독립적·주도적으로 처리할 수 있다. 본 보고서는 AI Agent의 핵심 기술 아키

In [7]:
for event in supervisor_agent.stream(
    {"messages": [HumanMessage(content="정보가 부족한 것 같은데, 추가 정보를 모아줘.")]},
    config=config,
    stream_mode="updates",
):
    for node_name, state in event.items():
        message = state["messages"][-1]
        print("=" * 60)
        print(f"Node: {node_name}")
        if getattr(message, "tool_calls", None):
            print(f"Tool calls: {[call['name'] for call in message.tool_calls]}")
        if message.text:
            print(message.text[:500])

Node: model
Tool calls: ['research']
Node: tools
AI 에이전트(AI Agent)의 핵심 프레임워크, 기존 RPA와의 비교, 온디바이스 SLM 활용, 보안 및 가드레일, 벤치마크, 그리고 산업별 활용 사례를 정리한 조사 보고서입니다.

---

# [종합 보고서] AI 에이전트(AI Agent) 기술 동향 및 산업 적용 실태

---

## 1. AI Agent 오픈소스 개발 프레임워크 비교 (LangGraph, CrewAI, AutoGen)

| 구분 | **LangGraph** (LangChain 생태계) | **CrewAI** | **AutoGen** (Microsoft) |
| :--- | :--- | :--- | :--- |
| **핵심 아키텍처** | **상태 기반 그래프 (Stateful Graph)**<br>노드(Node)=작업, 엣지(Edge)=조건부 흐름 | **역할 기반 협업 (Role-based Crew)**<br>인간의 조직/팀워크 시뮬레이션 | **대화 및 이벤트 기반 (Conversational & Eve
Node: model
Tool calls: ['write_report']
Node: tools
# AI Agent 기술 동향 보고서 (심화 심층판)

---

## [개요] 생성형 AI에서 에이전틱 AI(Agentic AI)로의 패러다임 전환

지난 2년간 생성형 AI 시장을 주도한 단일 거대언어모델(LLM) 중심의 '단발성 질의응답(Single-turn Chatbot)' 방식은 명확한 한계를 드러냈습니다. 복잡한 비즈니스 로직을 처리하거나 다단계 의사결정이 필요한 업무에서 기존 방식은 환각(Hallucination) 현상과 맥락 유지의 어려움을 겪었습니다.

이에 따라 최근 AI 패러다임은 **'에이전틱 AI(Agentic AI)'**로 급격히 전환되고 있습니다. 에이전틱 AI는 단순한 텍스트 생성을 넘어, **자율적 계획 수립(Planning), 장단기 기억(Memory), 도구 활용(Too

### Supervisor의 동작 방식

이 예제의 Supervisor도 필요한 작업을 판단하고, Tool을 실행하고, 결과를 관찰해 다음 행동을 결정하는 ReAct 방식으로 동작한다. 여기서 Tool은 독립적인 지시와 Context를 가진 Subagent를 호출한다. 예를 들어 자료 조사가 필요하면 `research`를 호출하고, 반환된 결과를 바탕으로 보고서 작성이나 검토를 위임한 뒤 최종 응답을 만든다.